# CoolData voxelization: docs vs triangulated

**Question:** does the voxelization of the CoolData docs differ from `CoolDataShard.voxelize` in speed and values?

**Answer:** _to be filled after the run_

- **Docs:** `VoxelFlowFieldDataset` of the `cooldata` package on the raw polyhedral mesh, bounds derived from the dataset min/max coordinates
  ([docs](https://cooldata.readthedocs.io/en/latest/usage/voxels.html))
- **Here:** `.triangulate()` first, grid points on the walls moved `WALL_INSET` inside, fixed domain `Grid.extent`
- The docs bounds differ slightly from `Grid.extent` (see the bounds column), so the value comparison includes that grid shift

In [ ]:
import contextlib
from concurrent.futures import ProcessPoolExecutor
import functools
import multiprocessing
from pathlib import Path
import tempfile
import time
from unittest import mock
import zipfile

import cooldata.pyvista_flow_field_dataset as cooldata_pyvista
from cooldata.pyvista_flow_field_dataset import PyvistaFlowFieldDataset, PyvistaSample
import cooldata.voxel_flow_field_dataset as cooldata_voxels
from cooldata.voxel_flow_field_dataset import VoxelFlowFieldDataset, VoxelFlowFieldDatasetConfig
from huggingface_hub import HfFileSystem
from IPython.display import Markdown, display
import numpy as np
import polars as pl

from coolfrog.engine.cooldata import HF_DATASET, CoolDataShard, CoolMetadata
from coolfrog.lib.config import HOST_CONFIG, Grid
from coolfrog.lib.datatypes import CHANNELS
from coolfrog.lib.experiment import markdown_table

N_DESIGNS = 100
BATCH_SIZE = 10  # designs per docs dataset, its samples stay on disk until the batch is compared: ~1.4 GB each at 1600x320x64
GRIDS = {"800x160x32": Grid(latent_scale=2), "1600x320x64": Grid(latent_scale=4)}
PATH_VOLUMES = HOST_CONFIG.cache_dir / "voxelization_comparison"  # `volume/` inside, the layout of `local_volume_path`
CHANNEL_NAMES = [channel.value for channel in CHANNELS]

## 1. Designs

The first 100 designs of the catalogue, downloaded once from Hugging Face so the benchmark excludes the download.

In [2]:
designs = CoolMetadata.load().designs.head(N_DESIGNS)
for design_id, run, batch in designs.select("design_id", "run", "batch").iter_rows():
    path = CoolDataShard.local_volume_path(PATH_VOLUMES, design_id)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with HfFileSystem().open(f"{HF_DATASET}/runs/{run}/{batch}.zip", "rb") as remote_zip, zipfile.ZipFile(remote_zip) as archive:
            archive.extract(path.name, path.parent)
print(f"{N_DESIGNS} designs in {PATH_VOLUMES}")

10 designs in /home/noob/git/cool-frog/.cache/voxelization_comparison


## 2. The two voxelizations

`voxelize_docs` is the constructor call of the docs: `VoxelFlowFieldDataset(cache_dir, VoxelFlowFieldDatasetConfig(pyvista_dataset, resolution))`.
It voxelizes the samples in parallel, writes them to disk and computes the normalization, all inside the timing.
The 100 designs do not fit on disk at 1600x320x64, so the docs dataset is built per batch of `BATCH_SIZE`.
The bounds are computed once over all designs with the docs `get_bounds`, as for one dataset, and enter the timing of every resolution.
`voxelize_triangulated_all` is the implementation of this repo with the same number of processes, reading the files and returning the arrays to the parent.
The library has no worker option, so `ProcessPoolExecutor` inside its voxel module is patched to `WORKERS` processes to bound memory.

In [3]:
WORKERS = 4  # the library default is all CPUs, too much memory at 1600x320x64


def pyvista_dataset_of(volume_paths: list[Path]) -> PyvistaFlowFieldDataset:
    """Docs raw dataset of the volumes."""
    return PyvistaFlowFieldDataset([PyvistaSample(path, path) for path in volume_paths])  # the surface path is never read


@contextlib.contextmanager
def forkserver_executors():
    """Run the `ProcessPoolExecutor`s of `cooldata` as `WORKERS` forkserver processes.

    - The library has no worker option, all CPUs use too much memory at 1600x320x64
    - A plain fork deadlocked once the parent had threads, e.g. from the Hugging Face downloads
    """
    executor = functools.partial(ProcessPoolExecutor, max_workers=WORKERS, mp_context=multiprocessing.get_context("forkserver"))
    with mock.patch.object(cooldata_voxels, "ProcessPoolExecutor", executor), mock.patch.object(cooldata_pyvista, "ProcessPoolExecutor", executor):
        yield


def voxelize_docs(volume_paths: list[Path], grid: Grid, cache_dir: str, bounds: tuple[float, ...]) -> VoxelFlowFieldDataset:
    """Docs `VoxelFlowFieldDataset` of the volumes with `WORKERS` processes, `bounds` replacing the bounds computed from the volumes."""
    config = VoxelFlowFieldDatasetConfig(pyvista_dataset=pyvista_dataset_of(volume_paths), resolution=grid.shape)
    with mock.patch.object(PyvistaFlowFieldDataset, "get_bounds", return_value=bounds), forkserver_executors():
        return VoxelFlowFieldDataset(cache_dir=cache_dir, config=config)


def docs_arrays(sample) -> tuple[np.ndarray, np.ndarray]:
    """(C, X, Y, Z) float32 fields in `CHANNELS` order and (X, Y, Z) fluid mask of a docs voxel sample."""
    data = sample.data
    velocity = data["Velocity"].numpy()
    fields = np.stack(
        [
            *(velocity[..., axis] for axis in range(3)),
            data["Pressure"].numpy(),
            data["Temperature"].numpy(),
            data["TurbulentKineticEnergy"].numpy(),
            data["TurbulentDissipationRate"].numpy(),
        ]
    )
    return fields, data["Mask"].numpy()


def voxelize_triangulated(volume_path: Path, grid: Grid) -> tuple[np.ndarray, np.ndarray]:
    """Implementation of this repo."""
    fields, mask = CoolDataShard.voxelize(volume_path, grid)
    return fields.numpy(), mask.numpy()


def voxelize_triangulated_all(design_rows: list[tuple[int, str, str]], grid: Grid) -> None:
    """Voxelize all designs of `PATH_VOLUMES` in `WORKERS` processes like `CoolData.build_voxel_cache` and drop the results.

    - `forkserver` as a plain fork deadlocked here once the parent had threads, e.g. from the Hugging Face downloads
    """
    voxelize_design = functools.partial(CoolDataShard.voxelize_design, grid=grid, source=PATH_VOLUMES)
    with multiprocessing.get_context("forkserver").Pool(WORKERS) as pool:
        for _ in pool.imap(voxelize_design, design_rows):
            pass


def timed(function, *arguments):
    """Result and wall time in s."""
    start = time.perf_counter()
    result = function(*arguments)
    return result, time.perf_counter() - start

## 3. Benchmark and comparison

Per resolution: time both on all designs, then compare design by design on the voxels that are fluid in both.
Mask mismatch is the share of voxels where only one method finds fluid.
Both are timed in separate processes, the pool of this repo first for all resolutions.

In [4]:
volume_paths = [CoolDataShard.local_volume_path(PATH_VOLUMES, design_id) for design_id in designs["design_id"]]
design_rows = designs.select("design_id", "run", "batch").rows()
seconds_triangulated = {}
for resolution, grid in GRIDS.items():
    _, seconds_triangulated[resolution] = timed(voxelize_triangulated_all, design_rows, grid)
    print(f"{resolution}: triangulated {seconds_triangulated[resolution]:.1f} s")
with forkserver_executors():
    bounds, seconds_bounds = timed(pyvista_dataset_of(volume_paths).get_bounds)
print(f"docs bounds {bounds} in {seconds_bounds:.1f} s")
speed_rows = []
rows = []
for resolution, grid in GRIDS.items():
    seconds_docs = seconds_bounds
    for batch_start in range(0, N_DESIGNS, BATCH_SIZE):
        batch = slice(batch_start, batch_start + BATCH_SIZE)
        with tempfile.TemporaryDirectory(dir=HOST_CONFIG.cache_dir) as cache_dir:  # /tmp can be a small RAM disk
            dataset_docs, seconds_batch = timed(voxelize_docs, volume_paths[batch], grid, cache_dir, bounds)
            seconds_docs += seconds_batch
            for design_id, path, sample in zip(designs["design_id"][batch], volume_paths[batch], dataset_docs.samples, strict=True):
                fields_docs, mask_docs = docs_arrays(sample)
                fields_triangulated, mask_triangulated = voxelize_triangulated(path, grid)
                both_fluid = mask_docs & mask_triangulated
                difference = fields_docs[:, both_fluid].astype(np.float64) - fields_triangulated[:, both_fluid]  # (C, N)
                rows.append(
                    {
                        "resolution": resolution,
                        "design_id": design_id,
                        "mask mismatch": float((mask_docs != mask_triangulated).mean()),
                        **{f"mse {name}": float(mse) for name, mse in zip(CHANNEL_NAMES, (difference**2).mean(axis=1), strict=True)},
                    }
                )
        print(f"{resolution}: designs {batch.start}-{batch.stop - 1} done, docs {seconds_docs:.0f} s so far")
    speed_rows.append(
        {
            "resolution": resolution,
            "docs [s]": seconds_docs,
            "triangulated [s]": seconds_triangulated[resolution],
            "docs bounds [m]": ", ".join(f"{bound:.5f}" for bound in bounds),
        }
    )
results = pl.DataFrame(rows)
speed = pl.DataFrame(speed_rows).with_columns((pl.col("docs [s]") / pl.col("triangulated [s]")).alias("speed-up"))

800x160x32 design 3: docs 34.4 s, triangulated 4.3 s


800x160x32 design 4: docs 32.6 s, triangulated 4.2 s


800x160x32 design 5: docs 32.0 s, triangulated 4.0 s


800x160x32 design 6: docs 35.4 s, triangulated 4.5 s


800x160x32 design 7: docs 36.7 s, triangulated 4.4 s


800x160x32 design 8: docs 40.0 s, triangulated 4.3 s


... (10 outputs left out)

1600x320x64 design 9: docs 295.6 s, triangulated 17.1 s


1600x320x64 design 14: docs 288.9 s, triangulated 16.4 s


1600x320x64 design 15: docs 200.3 s, triangulated 13.0 s


1600x320x64 design 16: docs 216.7 s, triangulated 13.9 s


### Speed

Wall time for all 100 designs per resolution, `WORKERS` processes each.

In [5]:
display(Markdown(markdown_table(speed)))

| resolution  | docs [s] | triangulated [s] | speed-up |
|-------------|----------|------------------|----------|
| 800x160x32  | 34.4     | 4.2              | 8.18     |
| 1600x320x64 | 262      | 15.5             | 16.9     |

### Values

Mean over the 100 designs of the MSE between both voxelizations in physical units, on voxels that are fluid in both.

In [6]:
mse = results.group_by("resolution", maintain_order=True).agg(pl.col("mask mismatch").mean(), pl.col(r"^mse .*$").mean())
display(Markdown(markdown_table(mse)))

| resolution  | mask mismatch | mse Vx [m/s] | mse Vy [m/s] | mse Vz [m/s] | mse p [Pa] | mse T [K] | mse k [m²/s²] | mse ε [m²/s³] |
|-------------|---------------|--------------|--------------|--------------|------------|-----------|---------------|---------------|
| 800x160x32  | 0.00573       | 0.00766      | 0.000844     | 0.000689     | 0.0364     | 0.12      | 0.00155       | 2.59e+04      |
| 1600x320x64 | 0.00299       | 0.0146       | 0.000922     | 0.000674     | 0.0552     | 0.147     | 0.00219       | 2e+04         |